In [ ]:
# install packages required

%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers
%pip install datasets
%pip install scikit-learn
%pip install 'accelerate>=1.1.0'
%pip install time
%pip install evaluate

In [1]:
# loading packages

import duckdb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import transformers
import accelerate
from transformers import BertForSequenceClassification
import time
import torch
import numpy as np
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification
import pickle
import os
import s3fs
from tqdm import tqdm
import random


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# loading the dataset

con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

query = "SELECT * FROM read_parquet('https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet')"
df = con.sql(query).df()

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   code    70000 non-null  object
 1   name    70000 non-null  object
 2   label   70000 non-null  object
dtypes: object(3)
memory usage: 1.6+ MB


In [4]:
#import the NACE

con = duckdb.connect(database=":memory:")

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

path_nace = 'https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/NACE_Rev2.1_Structure_Explanatory_Notes_EN.tsv'
query_definition = f"SELECT * FROM read_csv('{path_nace}')"
table = con.execute(query_definition).to_arrow_table()
nace = table.to_pylist()
nace[1]

nace2 = pd.DataFrame(nace)


In [5]:
nace2.head()

,ORDER_KEY,ID,CODE,HEADING,PARENT_ID,PARENT_CODE,LEVEL,Implementation_rule,Includes,IncludesAlso,Excludes
0,10,A,A,"AGRICULTURE, FORESTRY AND FISHING",None,None,1,None,This section includes the exploitation of vege...,This section also includes organic agriculture...,This section excludes undifferentiated subsist...
1,20,01,01,"Crop and animal production, hunting and relate...",A,A,2,"IMPLEMENTATION RULE \nIn agriculture, one freq...","This division includes two basic activities, n...",This division also includes service activities...,Agricultural activities exclude any subsequent...
2,30,011,01.1,Growing of non-perennial crops,01,01,3,None,This group includes the growing of non-perenni...,None,None
3,40,0111,01.11,"Growing of cereals, other than rice, leguminou...",011,01.1,4,None,This class includes all forms of growing of ce...,None,"This class excludes:\n- growing of rice, see 0..."
4,50,0112,01.12,Growing of rice,011,01.1,4,None,This class includes:\n- growing of rice,None,None


In [7]:
nace_filtered = nace2[nace2["CODE"].astype(str).str.len() == 5]
nace_filtered= nace_filtered[['CODE', 'HEADING','Includes', 'IncludesAlso', 'Excludes' ]]
nace_filtered["Includes"] = nace_filtered["Includes"].str.replace("\\n", "", regex=False)
nace_filtered["IncludesAlso"] = nace_filtered["IncludesAlso"].str.replace("\\n", "", regex=False)
nace_filtered["Excludes"] = nace_filtered["Excludes"].str.replace("\\n", "", regex=False)

In [8]:
nace_filtered.head()


,CODE,HEADING,Includes,IncludesAlso,Excludes
3,01.11,"Growing of cereals, other than rice, leguminou...",This class includes all forms of growing of ce...,None,"This class excludes:- growing of rice, see 01...."
4,01.12,Growing of rice,This class includes:- growing of rice,None,None
5,01.13,"Growing of vegetables and melons, roots and tu...",This class includes:- growing of leafy or stem...,This class also includes:- growing of sweetcorn,This class excludes:- growing of spices and ar...
6,01.14,Growing of sugar cane,None,None,"This class excludes:- growing of sugar beet, s..."
7,01.15,Growing of tobacco,This class includes:- growing of unmanufacture...,This class also includes:- preliminary process...,This class excludes:- preparation of tobacco l...


In [9]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""

    # convert in string
    text = str(text)

    # suppression  expressions 
    text = re.sub(r"This class include[s]?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"This class also includes?", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bnone\b", "", text, flags=re.IGNORECASE)

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # suppression spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# cleanning
nace_filtered["Includes_clean"] = nace_filtered["Includes"].apply(clean_text)
nace_filtered["IncludesAlso_clean"] = nace_filtered["IncludesAlso"].apply(clean_text)

#  knowledge creation
nace_filtered["knowledge"] = (
    nace_filtered["Includes_clean"] + " " +
    nace_filtered["IncludesAlso_clean"]
).str.strip()

# final dataset 

nace_kb = nace_filtered[["CODE", "HEADING", "knowledge"]].copy()
nace_kb["nace_text"] = (
    nace_kb["HEADING"].fillna("") + " " +
    nace_kb["knowledge"].fillna("")
)


In [10]:

nace_kb.head()


,CODE,HEADING,knowledge,nace_text
3,01.11,"Growing of cereals, other than rice, leguminou...",all forms of growing of cereals leguminous cro...,"Growing of cereals, other than rice, leguminou..."
4,01.12,Growing of rice,growing of rice,Growing of rice growing of rice
5,01.13,"Growing of vegetables and melons, roots and tu...",growing of leafy or stem vegetables e g artich...,"Growing of vegetables and melons, roots and tu..."
6,01.14,Growing of sugar cane,,Growing of sugar cane
7,01.15,Growing of tobacco,growing of unmanufactured tobacco preliminary ...,Growing of tobacco growing of unmanufactured t...


In [11]:
df = df.merge(
    nace_kb[["CODE", "nace_text"]],
    left_on="code",
    right_on="CODE",
    how="left"
)
df = df[["label", "nace_text", "code"]]

In [12]:
df

,label,nace_text,code
0,Pulses cultivation for market,"Growing of cereals, other than rice, leguminou...",01.11
1,Legume crop production activities,"Growing of cereals, other than rice, leguminou...",01.11
2,Broad bean farming operations,"Growing of cereals, other than rice, leguminou...",01.11
3,Chickpea harvesting and processing,"Growing of cereals, other than rice, leguminou...",01.11
4,Production of dried beans and peas,"Growing of cereals, other than rice, leguminou...",01.11
...,...,...,...
69995,Business document preparation services,Office administrative and support activities t...,82.10
69996,Letter and resume composition,Office administrative and support activities t...,82.10
69997,Administrative support on a fee basis,Office administrative and support activities t...,82.10
69998,Standard office support activities,Office administrative and support activities t...,82.10


In [13]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["code"],
    random_state=42
)

In [14]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [15]:
def tokenize(batch):
    user_enc = tokenizer(
        batch["label"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    nace_enc = tokenizer(
        batch["nace_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

    return {
        "input_ids_user": user_enc["input_ids"],
        "attention_mask_user": user_enc["attention_mask"],
        "input_ids_nace": nace_enc["input_ids"],
        "attention_mask_nace": nace_enc["attention_mask"],
    }

In [16]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)



# temporary dataset reduced
#train_dataset = train_dataset.shuffle().select(range(20))
#test_dataset = test_dataset.shuffle().select(range(4))


train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/56000 [00:00<?, ? examples/s]

Map:   2%|▏         | 1000/56000 [00:00<00:12, 4507.24 examples/s]

Map:   4%|▎         | 2000/56000 [00:00<00:09, 5422.97 examples/s]

Map:   5%|▌         | 3000/56000 [00:00<00:09, 5643.64 examples/s]

Map:   7%|▋         | 4000/56000 [00:00<00:08, 6116.77 examples/s]

...

Map:  79%|███████▊  | 11000/14000 [00:01<00:00, 6862.36 examples/s]

Map:  86%|████████▌ | 12000/14000 [00:01<00:00, 7235.29 examples/s]

Map:  93%|█████████▎| 13000/14000 [00:01<00:00, 7306.70 examples/s]

Map: 100%|██████████| 14000/14000 [00:02<00:00, 7439.79 examples/s]

Map: 100%|██████████| 14000/14000 [00:02<00:00, 6933.95 examples/s]

In [17]:
import torch.nn as nn
from transformers import BertModel

class SiameseBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = BertModel.from_pretrained("bert-base-uncased")
        self.projection = nn.Linear(self.encoder.config.hidden_size, 256)

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        return self.projection(cls)

    def encode_text(self, text, tokenizer, device):
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        return self.encode(enc["input_ids"], enc["attention_mask"])

In [18]:
# loss
class ContrastiveLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, sim, labels):
        # labels = 1 si match (toujours vrai ici)
        return ((1 - sim) ** 2).mean()

In [19]:
all_nace_texts = list(zip(nace_kb["nace_text"], nace_kb["CODE"]))
def get_negative(sample_code):
    while True:
        text, code = random.choice(all_nace_texts)
        if code != sample_code:
            return text

In [20]:
def encode_text(model, tokenizer, text, device):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    return model.encode(enc["input_ids"], enc["attention_mask"])

In [22]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [25]:
def tokenize(batch):
    user = tokenizer(batch["label"], padding=True, truncation=True, return_tensors="pt")
    nace = tokenizer(batch["nace_text"], padding=True, truncation=True, return_tensors="pt")

    return {
        "user_input_ids": user["input_ids"],
        "user_attention_mask": user["attention_mask"],
        "nace_input_ids": nace["input_ids"],
        "nace_attention_mask": nace["attention_mask"],
        "code": batch["code"]
    }

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2773269031.py, line 10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2868.31it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
model = SiameseBERT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(3):

    total_loss = 0
    loop = tqdm(train_df.itertuples(), total=len(train_df))

    for row in loop:

        user_text = row.label
        pos_text = row.nace_text
        neg_text = get_negative(row.code)

        optimizer.zero_grad()

        # TOKENIZATION ICI (SAFE)
        user = tokenizer(user_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        pos = tokenizer(pos_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        neg = tokenizer(neg_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        u = model.encode(user["input_ids"], user["attention_mask"])
        p = model.encode(pos["input_ids"], pos["attention_mask"])
        n = model.encode(neg["input_ids"], neg["attention_mask"])

        sim_pos = torch.cosine_similarity(u, p)
        sim_neg = torch.cosine_similarity(u, n)

        loss = torch.relu(sim_neg - sim_pos + 0.2).mean()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} loss: {total_loss:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3030.34it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/56000 [00:00<?, ?it/s]

  0%|          | 0/56000 [00:00<?, ?it/s, loss=0]

...

 21%|██        | 11543/56000 [30:46<1:54:18,  6.48it/s, loss=0]

 21%|██        | 11543/56000 [30:46<1:54:18,  6.48it/s, loss=0]

 21%|██        | 11544/56000 [30:46<1:56:08,  6.38it/s, loss=0]

 21%|██        | 11544/56000 [30:46<1:56:08,  6.38it/s, loss=0]

 21%|██        | 11545/56000 [30:46<1:54:57,  6.44it/s, loss=0]

In [ ]:
### saving model, tokenizer, encoder and nace
# local temporary dir
LOCAL_DIR = "./model_nace2"
os.makedirs(LOCAL_DIR, exist_ok=True)

# save local
trainer.save_model("./model_nace2")
tokenizer.save_pretrained("./model_nac2e")
with open("./model_nace2/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

#nomenclature.to_csv("./model_nace/nomenclature.csv", index=False)


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

trainer.save_model(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)

with open(f"{LOCAL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv(f"{LOCAL_DIR}/nomenclature.csv", index=False)

# -------------------
# upload to bucket
# -------------------

BUCKET_OUT = "thierry57"

files_to_upload = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "model.safetensors",        
    "label_encoder.pkl",
    "nomenclature.csv",
    "training_args.bin"
]

for file_name in files_to_upload:

    local_path = f"{LOCAL_DIR}/{file_name}"
    s3_path = f"{BUCKET_OUT}/model_nace/{file_name}"

    if os.path.exists(local_path):

        with open(local_path, "rb") as f_in:
            with fs.open(s3_path, "wb") as f_out:
                f_out.write(f_in.read())

        print(f"Uploaded: {s3_path}")


In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
import numpy as np

model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# 1. PRECOMPUTE NACE EMBEDDINGS
# =========================

nace_texts = nace_kb["HEADING"].tolist()
nace_codes = nace_kb["CODE"].tolist()

nace_embeddings = []

with torch.no_grad():
    for text in tqdm(nace_texts, desc="Encoding NACE"):

        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        ).to(device)

        emb = model.encode(enc["input_ids"], enc["attention_mask"])
        nace_embeddings.append(emb)

nace_embeddings = torch.cat(nace_embeddings, dim=0)  # (N, 256)


# =========================
# 2. EVALUATION LOOP
# =========================

top1 = 0
top5 = 0
mrr = 0

test_rows = test_df.itertuples()

with torch.no_grad():

    for row in tqdm(test_rows, total=len(test_df), desc="Evaluating"):

        text = row.label
        true_code = row.code

        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        ).to(device)

        query_emb = model.encode(enc["input_ids"], enc["attention_mask"])

        # cosine similarity avec tous les NACE
        sims = F.cosine_similarity(query_emb, nace_embeddings)

        # ranking
        topk = torch.topk(sims, k=5).indices.cpu().numpy()

        ranked_codes = [nace_codes[i] for i in topk]

        # TOP-1
        if ranked_codes[0] == true_code:
            top1 += 1

        # TOP-5
        if true_code in ranked_codes:
            top5 += 1

        # MRR
        rank = np.where(np.array(ranked_codes) == true_code)[0]
        if len(rank) > 0:
            mrr += 1 / (rank[0] + 1)


# =========================
# 3. METRICS
# =========================

n = len(test_df)

print("\n===== RESULTS =====")
print(f"Top-1 Accuracy: {top1 / n:.4f}")
print(f"Top-5 Accuracy: {top5 / n:.4f}")
print(f"MRR: {mrr / n:.4f}")

In [ ]:
from torch.utils.data import DataLoader

model = SiameseBERT()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = ContrastiveLoss()

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model.train()

for epoch in range(3):

    total_loss = 0

    loop = tqdm(train_df.itertuples(), total=len(train_df), desc=f"Epoch {epoch+1}")

    for row in loop:

        user_text = row.label
        pos_text = row.nace_text
        neg_text = get_negative(row.code)

        optimizer.zero_grad()

        u = encode_text(model, tokenizer, user_text, device)
        p = encode_text(model, tokenizer, pos_text, device)
        n = encode_text(model, tokenizer, neg_text, device)

        sim_pos = torch.cosine_similarity(u, p)
        sim_neg = torch.cosine_similarity(u, n)

        loss = torch.relu(sim_neg - sim_pos + 0.2).mean()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 🔥 affichage live loss
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} - total loss: {total_loss:.4f}")

In [ ]:
# label encoding
le = LabelEncoder()
df['target'] = le.fit_transform(df['code'])

num_labels = df['target'].nunique()
print(num_labels)

In [ ]:

### saving model, tokenizer, encoder and nace
# local temporary dir
LOCAL_DIR = "./model_nace2"
os.makedirs(LOCAL_DIR, exist_ok=True)

# save local
trainer.save_model("./model_nace2")
tokenizer.save_pretrained("./model_nac2e")
with open("./model_nace2/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

#nomenclature.to_csv("./model_nace/nomenclature.csv", index=False)


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

trainer.save_model(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)

with open(f"{LOCAL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv(f"{LOCAL_DIR}/nomenclature.csv", index=False)

# -------------------
# upload to bucket
# -------------------

BUCKET_OUT = "thierry57"

files_to_upload = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "model.safetensors",        
    "label_encoder.pkl",
    "nomenclature.csv",
    "training_args.bin"
]

for file_name in files_to_upload:

    local_path = f"{LOCAL_DIR}/{file_name}"
    s3_path = f"{BUCKET_OUT}/model_nace/{file_name}"

    if os.path.exists(local_path):

        with open(local_path, "rb") as f_in:
            with fs.open(s3_path, "wb") as f_out:
                f_out.write(f_in.read())

        print(f"Uploaded: {s3_path}")


In [ ]:
# create train and test dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['target'],  # class equilibtrate
    random_state=42
)

In [ ]:
# tokenize

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['label'],
        padding='max_length',
        truncation=True,
        max_length=128
    )


In [ ]:

# conversion in datasets
train_dataset = Dataset.from_pandas(train_df[['label', 'target']])
test_dataset = Dataset.from_pandas(test_df[['label', 'target']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("target", "labels")
test_dataset = test_dataset.rename_column("target", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


# temporary dataset reduced
#train_dataset = train_dataset.shuffle().select(range(20))
#test_dataset = test_dataset.shuffle().select(range(4))


In [ ]:

# charge bert model

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

In [ ]:
# trainning

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    disable_tqdm=False,   
    report_to="none"     
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

start = time.time()

trainer.train()

end = time.time()

print(f"Training time : {end - start:.2f} secondes")

### saving model, tokenizer, encoder and nace
# local temporary dir
LOCAL_DIR = "./model_nace"
os.makedirs(LOCAL_DIR, exist_ok=True)

# save local
trainer.save_model("./model_nace")
tokenizer.save_pretrained("./model_nace")
with open("./model_nace/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

#nomenclature.to_csv("./model_nace/nomenclature.csv", index=False)


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

trainer.save_model(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)

with open(f"{LOCAL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv(f"{LOCAL_DIR}/nomenclature.csv", index=False)

# -------------------
# upload to bucket
# -------------------

BUCKET_OUT = "thierry57"

files_to_upload = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "model.safetensors",        
    "label_encoder.pkl",
    "nomenclature.csv",
    "training_args.bin"
]

for file_name in files_to_upload:

    local_path = f"{LOCAL_DIR}/{file_name}"
    s3_path = f"{BUCKET_OUT}/model_nace/{file_name}"

    if os.path.exists(local_path):

        with open(local_path, "rb") as f_in:
            with fs.open(s3_path, "wb") as f_out:
                f_out.write(f_in.read())

        print(f"Uploaded: {s3_path}")


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_dataset)

In [ ]:
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

In [ ]:
y_pred_code = le.inverse_transform(y_pred)
y_true_code = le.inverse_transform(y_true)

In [ ]:
fs.ls("thierry57")

In [ ]:
# read file

BUCKET = "thierry57"
FILE_KEY_S3 = "naf_en_fr.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df_bpe = pd.read_csv(file_in, sep=";")

In [ ]:
#write file

BUCKET_OUT = "thierry57"
FILE_KEY_OUT_S3 = "nace_filtered.csv"
FILE_PATH_OUT_S3 = BUCKET_OUT + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    nace_filtered.to_csv(file_out)